In [ ]:
import pandas as pd
import numpy as np
import math
import glob
from scipy import ndimage as nd
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import cv2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Clase HandDP
clase para realizar el preprocesado de los datos de la base de datos. Se obtiene el template y los trazos realizados por el paciente.

In [ ]:
from cv2.cuda import EVENT_INTERPROCESS
class dataHandDP:
  def __init__(self, name_file):
    """
    @path: ruta de la imagen
    """
    self.path = f"/content/drive/MyDrive/{name_file}_out/"
    self.kernel_size = 5
    self.erosion_type = cv2.MORPH_RECT
    self.element_1 = cv2.getStructuringElement(cv2.MORPH_RECT, (2 * 1 + 1, 2 * 1 + 1), (1, 1))
    self.element_2 = cv2.getStructuringElement(cv2.MORPH_RECT, (2 * 2 + 1, 2 * 2 + 1), (2, 2))

  def preprocess_template(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    return cv2.medianBlur(img,self.kernel_size)

  def filters_TH(self, img, k_blur):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    img_blur = cv2.blur(img, (self.kernel_size,self.kernel_size))
    return cv2.medianBlur(img_blur, k_blur)

  def morph_TH(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    dilatation_dst = cv2.dilate(img, self.element_1)
    dilatation_dst = cv2.dilate(dilatation_dst, self.element_1)
    return cv2.erode(dilatation_dst, self.element_2)

  def pixel_treatments_TH(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
          pixels = img[y, x]
          if ((pixels[0] > 200 and pixels[1] > 200 and pixels[2] > 200)
              or pixels[0] < 70):
            img[y, x] = np.ones(3) * 255
    #
    return img

  def preprocess_TH(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    #
    img_tr_pixels = self.pixel_treatments_TH(img)
    img_filter = self.filters_TH(img_tr_pixels, 11)
    img_morph = self.morph_TH(img_filter)
    #
    return img_morph

  def get_template(self, pixels):
    """
    @pixels: píxeles de la imagen
    @return: valor del píxel
    """
    result = np.ones(3)*255
    if pixels[0] < 100 and pixels[1] < 100 and pixels[2] < 100:
      result = np.zeros(3)
    #
    return result

  def get_THimg(self, pixels):
    """
    @pixels: píxeles de la imagen
    @return: valor del píxel BGR
    """
    result = pixels
    if abs(pixels[0] - pixels[1]) < 30 and abs(pixels[0] - pixels[2]) < 30 \
          and abs(pixels[1] - pixels[2]) < 40:
      result = np.ones(3)*255
    #
    return result

  def save_img(self, id_img, imgTemplate, imgTH):
    """
    @imgTemplate: imagen preprocesada
    @imgTH: imagen preprocesada
    @return: imagen preprocesada
    """
    cv2.imwrite(f"{self.path}{id_img}_template.jpg", imgTemplate)
    cv2.imwrite(f"{self.path}{id_img}_TH.jpg", imgTH)

  def read_img(self, path):
    """
    @path: ruta de la imagen
    @return: imagen preprocesada
    """
    img = cv2.imread(path, cv2.COLOR_BGR2RGB)
    imgTemplate = self.preprocess_template(img.copy())
    imgTH = self.preprocess_TH(img.copy())
    # Recorrer píxeles
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            pixel_template = self.get_template(imgTemplate[y, x])
            pixel_TH = self.get_THimg(imgTH[y, x])
            # Almacenar nuevos valores
            imgTemplate[y, x] = pixel_template
            imgTH[y, x] = pixel_TH
    #
    imgTH = self.filters_TH(imgTH, 3)
    return imgTemplate, imgTH


  def run(self, path):
    """
    @path: ruta de la imagen
    @return: imagen preprocesada
    """
    for v_path in glob.glob(f"{path}*.jpg"):
      print("Process:", v_path)
      id_img = v_path.split("/")[-1].split(".")[0]
      imgTemplate, imgTH = self.read_img(v_path)
      self.save_img(id_img, imgTemplate, imgTH)


#Ejecución

In [ ]:
list_exe = ["Spiral","Control"),("Spiral","Patients")
    ("Meander","Control"),("Meander","Patients")]
for (name, group) in list_exe:
  dp = dataHandDP(f"{name}{group}")
  dp.run(f"/content/drive/MyDrive/{name}{group}/")

Process: /content/drive/MyDrive/MeanderControl/0068-8.jpg


/tmp/ipykernel_13114/153230861.py:81: RuntimeWarning: overflow encountered in scalar subtract
  and abs(pixels[1] - pixels[2]) < 40:
/tmp/ipykernel_13114/153230861.py:80: RuntimeWarning: overflow encountered in scalar subtract
  if abs(pixels[0] - pixels[1]) < 30 and abs(pixels[0] - pixels[2]) < 30 \


Process: /content/drive/MyDrive/MeanderControl/0092-7.jpg
Process: /content/drive/MyDrive/MeanderControl/0068-5.jpg
Process: /content/drive/MyDrive/MeanderControl/0098-8.jpg
Process: /content/drive/MyDrive/MeanderControl/0092-8.jpg
Process: /content/drive/MyDrive/MeanderControl/0098-5.jpg
Process: /content/drive/MyDrive/MeanderControl/0098-6.jpg
Process: /content/drive/MyDrive/MeanderControl/0068-7.jpg
Process: /content/drive/MyDrive/MeanderControl/0068-6.jpg
Process: /content/drive/MyDrive/MeanderControl/0098-7.jpg
Process: /content/drive/MyDrive/MeanderControl/0092-6.jpg
Process: /content/drive/MyDrive/MeanderControl/0092-5.jpg
Process: /content/drive/MyDrive/MeanderControl/0100-6.jpg
Process: /content/drive/MyDrive/MeanderControl/0100-8.jpg
Process: /content/drive/MyDrive/MeanderControl/0112-6.jpg
Process: /content/drive/MyDrive/MeanderControl/0106-7.jpg
Process: /content/drive/MyDrive/MeanderControl/0104-7.jpg
Process: /content/drive/MyDrive/MeanderControl/0102-7.jpg
Process: /cont